# Step 5: Classification-Enhanced RAG Pipeline

Notebook 03 built a RAG pipeline that retrieves services using **semantic similarity only**. This notebook adds the multi-label classifier from notebook 04 to:

1. **Understand intent** — classify what types of services the user is asking about
2. **Improve retrieval** — use classification to filter or re-rank search results
3. **Improve generation** — give the LLM explicit knowledge of detected service types

**Three retrieval strategies compared:**
- **Baseline** — pure semantic search (notebook 03 approach)
- **Filtered** — semantic search + ChromaDB metadata filter using classifier output
- **Hybrid (recommended)** — retrieve large candidate set, re-rank with type-match scoring

**Make sure Ollama is running:** Open a terminal and run `ollama serve`

## Install Required Packages

In [1]:
!pip install ollama sentence-transformers chromadb transformers torch -q

## Load All Components

In [2]:
import json
import numpy as np
import torch
import chromadb
import ollama
import warnings
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings('ignore')

In [3]:
# 1. Embedding model (same as notebooks 02 and 03)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

# 2. ChromaDB (created in notebook 02)
chroma_client = chromadb.PersistentClient(path='../data/chroma_db')
collection = chroma_client.get_collection('services')
print(f"ChromaDB connected: {collection.count()} services")

# 3. Classifier model (trained in notebook 04)
clf_model_path = '../models/service_type_classifier_v2_final'
clf_tokenizer = AutoTokenizer.from_pretrained(clf_model_path)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_model_path)
clf_model.eval()
print("Classifier loaded!")

# 4. Label mappings
with open(f'{clf_model_path}/label_mappings.json') as f:
    label_data = json.load(f)
SERVICE_TYPES = label_data['service_types']
label2id = label_data['label2id']
id2label = {int(k): v for k, v in label_data['id2label'].items()}
NUM_LABELS = len(SERVICE_TYPES)
print(f"Labels loaded: {NUM_LABELS} service types")

# 5. Optimal per-label thresholds
with open(f'{clf_model_path}/optimal_thresholds.json') as f:
    threshold_data = json.load(f)
optimal_thresholds = np.array([threshold_data[t] for t in SERVICE_TYPES])
print(f"Thresholds loaded (range: {optimal_thresholds.min():.2f} - {optimal_thresholds.max():.2f})")

Embedding model loaded!
ChromaDB connected: 1719 services
Classifier loaded!
Labels loaded: 22 service types
Thresholds loaded (range: 0.25 - 0.85)


In [4]:
# 6. Test Ollama connection
try:
    response = ollama.chat(model='llama3.2', messages=[
        {'role': 'user', 'content': 'Say "Hello, I am ready!" and nothing else.'}
    ])
    print(f"Ollama ready: {response['message']['content']}")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure Ollama is running: open a terminal and run 'ollama serve'")

Ollama ready: Hello, I am ready!


## Core Functions

In [5]:
def classify_query(query, threshold_scale=0.7):
    """
    Classify a user query to detect service type intent.
    Uses scaled-down thresholds (0.7x) because queries are shorter
    and less specific than full service descriptions.
    
    Returns:
        dict with 'types', 'probabilities', and 'all_probs'
    """
    inputs = clf_tokenizer(
        query, return_tensors='pt', truncation=True,
        max_length=512, padding=True
    )
    
    with torch.no_grad():
        outputs = clf_model(**inputs)
        probs = torch.sigmoid(outputs.logits).squeeze().numpy()
    
    scaled_thresholds = optimal_thresholds * threshold_scale
    
    detected = []
    probabilities = {}
    for i in range(NUM_LABELS):
        probabilities[SERVICE_TYPES[i]] = float(probs[i])
        if probs[i] >= scaled_thresholds[i]:
            detected.append(SERVICE_TYPES[i])
    
    return {
        'types': detected,
        'probabilities': probabilities,
        'all_probs': probs
    }


# Quick test
test = classify_query("homeless veteran needs emergency shelter")
print("Detected types:")
for t in test['types']:
    print(f"  [{test['probabilities'][t]:.2f}] {t}")

Detected types:
  [0.87] Emergency Shelter & Crisis Intervention
  [0.88] Homelessness Prevention & Diversion
  [0.34] Food & Basic Needs Assistance


In [6]:
def search_services_basic(query, n_results=5):
    """Baseline: pure semantic search (same as notebook 03)."""
    query_embedding = embedding_model.encode(query).tolist()
    return collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )


def search_services_filtered(query, detected_types, n_results=5):
    """Filtered: semantic search + metadata type filter."""
    query_embedding = embedding_model.encode(query).tolist()
    
    if not detected_types:
        return collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
    
    # Build where clause: match ANY of the detected types
    if len(detected_types) == 1:
        where_clause = {'types': {'$contains': detected_types[0]}}
    else:
        where_clause = {
            '$or': [{'types': {'$contains': t}} for t in detected_types]
        }
    
    try:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where_clause
        )
    except Exception as e:
        print(f"Filter failed ({e}), falling back to basic search")
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
    
    return results


print("Search functions ready!")

Search functions ready!


## Hybrid Strategy: Retrieve More, Then Re-rank

Pure filtering is aggressive — if the classifier misses a relevant type, the document is excluded entirely. The hybrid approach retrieves a larger candidate set (20) using semantic search, then re-ranks by combining semantic similarity with a type-match bonus score.

In [7]:
def compute_type_match_score(doc_types_str, query_probs):
    """
    Score how well a document's types match the query's classified intent.
    Returns the max classifier probability across matching types (0-1).
    """
    if not doc_types_str:
        return 0.0
    
    doc_types = {t.strip() for t in doc_types_str.split(',')}
    
    max_prob = 0.0
    for t in doc_types:
        if t in query_probs:
            max_prob = max(max_prob, query_probs[t])
    
    return max_prob


def search_services_hybrid(query, n_results=5, n_candidates=20,
                           semantic_weight=0.6, type_weight=0.4):
    """
    Hybrid: retrieve large candidate set, re-rank with type-match scoring.
    
    1. Classify query to get type probabilities
    2. Retrieve n_candidates via semantic search
    3. Score each: combined = semantic_weight * sim + type_weight * type_match
    4. Return top n_results by combined score
    """
    # Classify the query
    classification = classify_query(query)
    query_probs = classification['probabilities']
    
    # Retrieve candidates
    candidates = search_services_basic(query, n_results=n_candidates)
    
    # Score each candidate
    distances = candidates['distances'][0]
    
    # Convert distances to similarity scores (0-1)
    sim_scores = [1 / (1 + d) for d in distances]
    if max(sim_scores) > min(sim_scores):
        sim_min, sim_max = min(sim_scores), max(sim_scores)
        sim_normalized = [(s - sim_min) / (sim_max - sim_min) for s in sim_scores]
    else:
        sim_normalized = [1.0] * len(sim_scores)
    
    # Compute type-match scores
    type_scores = []
    for meta in candidates['metadatas'][0]:
        score = compute_type_match_score(meta.get('types', ''), query_probs)
        type_scores.append(score)
    
    # Combined scores
    combined = [
        semantic_weight * sim + type_weight * ts
        for sim, ts in zip(sim_normalized, type_scores)
    ]
    
    # Sort by combined score (descending)
    ranked_indices = np.argsort(combined)[::-1][:n_results]
    
    # Build results in ChromaDB format
    results = {
        'ids': [[candidates['ids'][0][i] for i in ranked_indices]],
        'documents': [[candidates['documents'][0][i] for i in ranked_indices]],
        'metadatas': [[candidates['metadatas'][0][i] for i in ranked_indices]],
        'distances': [[candidates['distances'][0][i] for i in ranked_indices]],
        'scores': [combined[i] for i in ranked_indices],
        'detected_types': classification['types'],
    }
    
    return results


# Quick test
test_results = search_services_hybrid("homeless veteran needs shelter")
print(f"Detected types: {test_results['detected_types']}")
print(f"\nTop {len(test_results['metadatas'][0])} services (re-ranked):")
for i, meta in enumerate(test_results['metadatas'][0]):
    print(f"  {i+1}. {meta['service_name']}")
    print(f"     Types: {meta.get('types', 'N/A')[:80]}")

Detected types: ['Emergency Shelter & Crisis Intervention', 'Homelessness Prevention & Diversion', 'Transitional & Supportive Housing']

Top 5 services (re-ranked):
  1. National Call Center for Homeless Veterans
     Types: Mental Health Services, Veteran Services
  2. Coordinated Entry Access Site (CES), VA Healthcare Systems, Oceanside
     Types: Homelessness Prevention & Diversion, Disability Services, Case Management & Coor
  3. Veteran's Transitional Housing Program
     Types: Transitional & Supportive Housing, Emergency Shelter & Crisis Intervention
  4. Supportive Services for Veteran Families (SSVF)
     Types: Case Management & Coordination, Emergency Shelter & Crisis Intervention, Disabil
  5. Welcome Home Family Program
     Types: Transitional & Supportive Housing, Substance Abuse Disorder, Emergency Shelter &


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## LLM Response Generation

In [8]:
def format_services_for_llm(search_results):
    """Format search results into a context string for the LLM."""
    services_text = ""
    for i, (meta, doc) in enumerate(zip(
        search_results['metadatas'][0], search_results['documents'][0]
    )):
        services_text += f"""
---
SERVICE {i+1}: {meta.get('service_name', 'Unknown')}
Organization: {meta.get('organization', 'N/A')}
Phone: {meta.get('phone', 'N/A')}
Address: {meta.get('address', 'N/A')}
Types: {meta.get('types', 'N/A')}
Area Served: {meta.get('area_served', 'N/A')}

Full Details:
{doc[:1500]}
"""
    return services_text


print("Formatting function ready!")

Formatting function ready!


In [9]:
def ask_basic(query, n_results=5):
    """Baseline RAG: semantic search + LLM (same as notebook 03)."""
    search_results = search_services_basic(query, n_results=n_results)
    services_context = format_services_for_llm(search_results)
    
    system_prompt = """You are a helpful assistant for case managers working with homeless and at-risk populations in San Diego.

Your job is to:
1. Analyze the services provided in the context
2. Recommend the most relevant services for the client's situation
3. Explain eligibility requirements clearly
4. Provide contact information and next steps
5. Note any important details (hours, documents needed, etc.)

Be concise but thorough. If a service doesn't seem like a good match, say so.
Always prioritize the client's immediate needs."""

    user_message = f"""A case manager is asking: \"{query}\"

Here are the relevant services from our database:
{services_context}

Based on these services, provide helpful recommendations for the case manager."""

    response = ollama.chat(
        model='llama3.2',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ]
    )
    
    return {
        'answer': response['message']['content'],
        'services_found': [m['service_name'] for m in search_results['metadatas'][0]],
        'detected_types': [],
        'method': 'basic'
    }


def ask_enhanced(query, n_results=5):
    """Enhanced RAG: classify query + hybrid retrieval + intent-aware LLM prompt."""
    # Hybrid retrieval (includes classification internally)
    search_results = search_services_hybrid(query, n_results=n_results)
    detected_types = search_results['detected_types']
    services_context = format_services_for_llm(search_results)
    
    # Build intent-aware prompt
    if detected_types:
        types_str = ', '.join(detected_types)
        intent_section = f"""\nThe client's query has been analyzed and the following service needs were detected:
{types_str}

Prioritize services that match these detected needs."""
    else:
        intent_section = ""
    
    system_prompt = f"""You are a helpful assistant for case managers working with homeless and at-risk populations in San Diego.
{intent_section}
Your job is to:
1. Analyze the services provided in the context
2. Recommend the most relevant services for the client's situation
3. Explain eligibility requirements clearly
4. Provide contact information and next steps
5. Note any important details (hours, documents needed, etc.)

Be concise but thorough. If a service doesn't match the detected needs well, say so and explain why it might still be useful.
Always prioritize the client's immediate needs."""

    user_message = f"""A case manager is asking: \"{query}\"

Here are the relevant services from our database:
{services_context}

Based on these services, provide helpful recommendations for the case manager."""

    response = ollama.chat(
        model='llama3.2',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ]
    )
    
    return {
        'answer': response['message']['content'],
        'services_found': [m['service_name'] for m in search_results['metadatas'][0]],
        'detected_types': detected_types,
        'method': 'enhanced'
    }


print("Pipeline functions ready!")

Pipeline functions ready!


## Comparison: Basic RAG vs Classification-Enhanced RAG

Let's compare the two approaches on the same queries to see how classification improves results.

In [10]:
test_queries = [
    "I have a homeless veteran who needs emergency shelter tonight",
    "Single mother with 2 kids facing eviction, needs rent assistance",
    "Senior citizen needs help with food and utilities",
    "Young adult aging out of foster care needs housing and mental health support",
    "Person fleeing domestic violence needs a safe place to stay",
]

print(f"Test queries: {len(test_queries)}")
for i, q in enumerate(test_queries):
    print(f"  {i+1}. {q}")

Test queries: 5
  1. I have a homeless veteran who needs emergency shelter tonight
  2. Single mother with 2 kids facing eviction, needs rent assistance
  3. Senior citizen needs help with food and utilities
  4. Young adult aging out of foster care needs housing and mental health support
  5. Person fleeing domestic violence needs a safe place to stay


In [11]:
# Run both pipelines on all queries (retrieval only, no LLM yet)
print("Retrieval Comparison: Basic vs Enhanced")
print("=" * 80)

for q in test_queries:
    # Classify
    classification = classify_query(q)
    
    # Basic retrieval
    basic_results = search_services_basic(q, n_results=5)
    basic_names = [m['service_name'] for m in basic_results['metadatas'][0]]
    
    # Hybrid retrieval
    hybrid_results = search_services_hybrid(q, n_results=5)
    hybrid_names = [m['service_name'] for m in hybrid_results['metadatas'][0]]
    
    print(f"\nQuery: {q}")
    print(f"Detected types: {classification['types']}")
    print(f"\n  {'Basic':40} | {'Hybrid (re-ranked)':40}")
    print(f"  {'-'*40} | {'-'*40}")
    
    for i in range(5):
        b = basic_names[i][:38] if i < len(basic_names) else ''
        h = hybrid_names[i][:38] if i < len(hybrid_names) else ''
        # Mark if service moved into top 5 from hybrid
        marker = ' *' if h and hybrid_names[i] not in basic_names else ''
        print(f"  {i+1}. {b:38} | {i+1}. {h:38}{marker}")
    
    # Show differences
    new_services = set(hybrid_names) - set(basic_names)
    if new_services:
        print(f"  * New in hybrid: {', '.join(list(new_services)[:2])}")
    
    print("-" * 80)

Retrieval Comparison: Basic vs Enhanced

Query: I have a homeless veteran who needs emergency shelter tonight
Detected types: ['Emergency Shelter & Crisis Intervention', 'Homelessness Prevention & Diversion', 'Food & Basic Needs Assistance']

  Basic                                    | Hybrid (re-ranked)                      
  ---------------------------------------- | ----------------------------------------
  1. National Call Center for Homeless Vete | 1. Harm Reduction Shelter                
  2. Harm Reduction Shelter                 | 2. Emergency Adult Shelter VVSD          
  3. Emergency Adult Shelter VVSD           | 3. Emergency Adult Shelter VVSD          
  4. Emergency Adult Shelter VVSD           | 4. National Call Center for Homeless Vete
  5. Homeless Veterans' Reintegration Progr | 5. Supportive Services for Veteran Famili *
  * New in hybrid: Supportive Services for Veteran Families (SSVF)
----------------------------------------------------------------------------

In [12]:
# Full LLM comparison on one query
comparison_query = "I have a homeless veteran who needs emergency shelter tonight"

print(f"QUERY: {comparison_query}")
print("=" * 80)

# Basic pipeline
print("\n--- BASIC RAG (semantic search only) ---")
basic_result = ask_basic(comparison_query)
print(f"\nServices found: {basic_result['services_found']}")
print(f"\n{basic_result['answer']}")

print("\n" + "=" * 80)

# Enhanced pipeline
print("\n--- ENHANCED RAG (classification + hybrid + intent-aware prompt) ---")
enhanced_result = ask_enhanced(comparison_query)
print(f"\nDetected types: {enhanced_result['detected_types']}")
print(f"Services found: {enhanced_result['services_found']}")
print(f"\n{enhanced_result['answer']}")

QUERY: I have a homeless veteran who needs emergency shelter tonight

--- BASIC RAG (semantic search only) ---

Services found: ['National Call Center for Homeless Veterans', 'Harm Reduction Shelter', 'Emergency Adult Shelter VVSD', 'Emergency Adult Shelter VVSD', "Homeless Veterans' Reintegration Program"]

**Recommendations:**

Considering the client is a homeless veteran, I recommend **SERVICE 1: National Call Center for Homeless Veterans (VA)** as the first point of contact. The VA provides mental health services and veteran-specific support, which could help address the veteran's immediate needs.

If the national call center determines that the veteran requires more intensive support, they can refer them to other services within the network or provide referrals outside the VA system.

**Alternative Recommendations:**

- If the client has co-occurring substance use issues (as per Service 2's eligibility criteria), **SERVICE 2: Harm Reduction Shelter** could be a suitable option. Ho

In [13]:
# Three-way comparison: Basic vs Filtered vs Hybrid
print("Three-Way Retrieval Comparison")
print("=" * 90)

for q in test_queries[:3]:
    classification = classify_query(q)
    
    basic = search_services_basic(q, n_results=5)
    filtered = search_services_filtered(q, classification['types'], n_results=5)
    hybrid = search_services_hybrid(q, n_results=5)
    
    basic_names = [m['service_name'][:25] for m in basic['metadatas'][0]]
    filtered_names = [m['service_name'][:25] for m in filtered['metadatas'][0]]
    hybrid_names = [m['service_name'][:25] for m in hybrid['metadatas'][0]]
    
    print(f"\nQuery: {q[:70]}")
    print(f"Types: {classification['types']}")
    print(f"\n  {'#':3} {'Basic':27} {'Filtered':27} {'Hybrid':27}")
    print(f"  {'---':3} {'-'*27} {'-'*27} {'-'*27}")
    for i in range(5):
        b = basic_names[i] if i < len(basic_names) else '-'
        f = filtered_names[i] if i < len(filtered_names) else '-'
        h = hybrid_names[i] if i < len(hybrid_names) else '-'
        print(f"  {i+1:3} {b:27} {f:27} {h:27}")
    print("-" * 90)

Three-Way Retrieval Comparison
Filter failed (Expected where operator to be one of $gt, $gte, $lt, $lte, $ne, $eq, $in, $nin, got $contains in query.), falling back to basic search

Query: I have a homeless veteran who needs emergency shelter tonight
Types: ['Emergency Shelter & Crisis Intervention', 'Homelessness Prevention & Diversion', 'Food & Basic Needs Assistance']

  #   Basic                       Filtered                    Hybrid                     
  --- --------------------------- --------------------------- ---------------------------
    1 National Call Center for    National Call Center for    Harm Reduction Shelter     
    2 Harm Reduction Shelter      Harm Reduction Shelter      Emergency Adult Shelter V  
    3 Emergency Adult Shelter V   Emergency Adult Shelter V   Emergency Adult Shelter V  
    4 Emergency Adult Shelter V   Emergency Adult Shelter V   National Call Center for   
    5 Homeless Veterans' Reinte   Homeless Veterans' Reinte   Supportive Services for

## Understanding the Re-ranking Process

Let's look under the hood at how hybrid re-ranking works for a single query.

In [14]:
# Detailed re-ranking visualization
debug_query = "Person fleeing domestic violence needs a safe place to stay"

classification = classify_query(debug_query)
print(f"Query: {debug_query}")
print(f"Detected types: {classification['types']}")
print()

# Get 20 candidates
candidates = search_services_basic(debug_query, n_results=20)
distances = candidates['distances'][0]

# Compute scores
sim_scores = [1 / (1 + d) for d in distances]
sim_min, sim_max = min(sim_scores), max(sim_scores)
sim_normalized = [(s - sim_min) / (sim_max - sim_min) for s in sim_scores]

type_scores = [
    compute_type_match_score(m.get('types', ''), classification['probabilities'])
    for m in candidates['metadatas'][0]
]

combined = [0.6 * s + 0.4 * t for s, t in zip(sim_normalized, type_scores)]
new_rank = np.argsort(combined)[::-1]

print(f"{'Orig':>4} {'New':>4}  {'Semantic':>8} {'TypeMatch':>9} {'Combined':>8}  Service Name")
print("-" * 90)
for new_pos, orig_idx in enumerate(new_rank):
    name = candidates['metadatas'][0][orig_idx]['service_name'][:40]
    moved = '' if new_pos == orig_idx else f' ({"^" if new_pos < orig_idx else "v"}{abs(new_pos - orig_idx)})'
    print(f"  {orig_idx+1:>2}  {new_pos+1:>2}    {sim_normalized[orig_idx]:.3f}     {type_scores[orig_idx]:.3f}    {combined[orig_idx]:.3f}  {name}{moved}")

Query: Person fleeing domestic violence needs a safe place to stay
Detected types: ['Emergency Shelter & Crisis Intervention', 'Domestic Violence Support', 'Family Services', 'Mental Health Services']

Orig  New  Semantic TypeMatch Combined  Service Name
------------------------------------------------------------------------------------------
   1   1    1.000     0.946    0.979  Confronting Domestic Violence
   2   2    0.864     0.946    0.897  One Safe Place
   3   3    0.819     0.946    0.870  One Safe Place
   4   4    0.649     0.946    0.768  Domestic Violence Emergency Shelter
   5   5    0.624     0.881    0.727  New Journey, Transitional Housing Progra
   6   6    0.562     0.946    0.716  Domestic Violence Emergency Shelter
   7   7    0.368     0.946    0.599  Carol's House
   8   8    0.345     0.946    0.585  Carol's House
  10   9    0.279     0.946    0.546  National Domestic Violence Hotline (^1)
  11  10    0.247     0.946    0.527  Crisis Line (^1)
  12  11    0.19

In [15]:
# Edge cases: what happens with tricky queries?
edge_cases = [
    "help with anything available",
    "need a place to sleep",
    "refugee family needs everything",
]

print("Edge Case Analysis")
print("=" * 70)

for q in edge_cases:
    classification = classify_query(q)
    hybrid = search_services_hybrid(q, n_results=3)
    
    print(f"\nQuery: \"{q}\"")
    print(f"  Detected types: {classification['types'] if classification['types'] else '(none)'}")
    print(f"  Top services:")
    for i, m in enumerate(hybrid['metadatas'][0]):
        print(f"    {i+1}. {m['service_name']}")
    print("-" * 70)

Edge Case Analysis

Query: "help with anything available"
  Detected types: ['Food & Basic Needs Assistance', 'Mental Health Services']
  Top services:
    1. Access to Independence, San Diego
    2. Access to Independence, San Diego
    3. Family Resource Center, Beacon
----------------------------------------------------------------------

Query: "need a place to sleep"
  Detected types: ['Emergency Shelter & Crisis Intervention', 'Homelessness Prevention & Diversion', 'Transitional & Supportive Housing', 'Family Services', 'Veteran Services', 'Refugee Services', 'Food & Basic Needs Assistance', 'Mental Health Services']
  Top services:
    1. Safe Sleeping Program
    2. Women and Children Shelter Services
    3. Sober Living Homes
----------------------------------------------------------------------

Query: "refugee family needs everything"
  Detected types: ['Emergency Shelter & Crisis Intervention', 'Homelessness Prevention & Diversion', 'Family Services', 'Food & Basic Needs As

## Putting It All Together

In [16]:
def ask_case_manager(query, method='hybrid', n_results=5, verbose=True):
    """
    Main entry point for the classification-enhanced RAG pipeline.
    
    Args:
        query: User query string
        method: 'basic', 'filtered', or 'hybrid' (default)
        n_results: Number of services to retrieve
        verbose: Print intermediate steps
    
    Returns:
        dict with 'answer', 'services_found', 'detected_types', 'method'
    """
    if verbose:
        print(f"Query: {query}")
        print(f"Method: {method}")
        print("=" * 70)
    
    # Classify query (for all methods except basic)
    detected_types = []
    if method != 'basic':
        classification = classify_query(query)
        detected_types = classification['types']
        if verbose:
            print(f"Detected service types: {detected_types}")
    
    # Retrieve services
    if method == 'basic':
        search_results = search_services_basic(query, n_results=n_results)
    elif method == 'filtered':
        search_results = search_services_filtered(query, detected_types, n_results=n_results)
    else:  # hybrid
        search_results = search_services_hybrid(query, n_results=n_results)
        detected_types = search_results.get('detected_types', detected_types)
    
    services_context = format_services_for_llm(search_results)
    
    if verbose:
        print(f"\nServices retrieved:")
        for m in search_results['metadatas'][0]:
            print(f"  - {m['service_name']}")
    
    # Build prompt
    if method != 'basic' and detected_types:
        types_str = ', '.join(detected_types)
        intent_section = f"""\nThe client's query has been analyzed and the following service needs were detected:
{types_str}

Prioritize services that match these detected needs."""
    else:
        intent_section = ""
    
    system_prompt = f"""You are a helpful assistant for case managers working with homeless and at-risk populations in San Diego.
{intent_section}
Your job is to:
1. Analyze the services provided in the context
2. Recommend the most relevant services for the client's situation
3. Explain eligibility requirements clearly
4. Provide contact information and next steps
5. Note any important details (hours, documents needed, etc.)

Be concise but thorough. If a service doesn't match the detected needs well, say so and explain why it might still be useful.
Always prioritize the client's immediate needs."""

    user_message = f"""A case manager is asking: \"{query}\"

Here are the relevant services from our database:
{services_context}

Based on these services, provide helpful recommendations for the case manager."""

    response = ollama.chat(
        model='llama3.2',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ]
    )
    
    if verbose:
        print(f"\n{'='*70}")
        print("RESPONSE:")
        print("=" * 70)
        print(response['message']['content'])
    
    return {
        'answer': response['message']['content'],
        'services_found': [m['service_name'] for m in search_results['metadatas'][0]],
        'detected_types': detected_types,
        'method': method
    }


print("ask_case_manager() ready!")

ask_case_manager() ready!


In [17]:
# Interactive mode - change this query and run the cell
your_query = "I need help finding housing for a disabled veteran"

result = ask_case_manager(your_query, method='hybrid', verbose=True)

Query: I need help finding housing for a disabled veteran
Method: hybrid
Detected service types: ['Emergency Shelter & Crisis Intervention', 'Homelessness Prevention & Diversion', 'Transitional & Supportive Housing', 'Disability Services']

Services retrieved:
  - Transitional Housing
  - Adjoin Veterans SSVF Rapid Re-Housing, San Diego
  - Coordinated Entry Access Site (CES), VA Healthcare Systems, Oceanside
  - Veteran's Transitional Housing Program
  - Welcome Home Family Program

RESPONSE:
Based on the detected needs of finding housing for a disabled veteran, I recommend the following services:

1. **Transitional Housing**: Wounded Warrior Homes (SERVICE 1) is an excellent option for a disabled veteran. This organization provides transitional housing, case management, independent living support, meals, peer support, and life skills training specifically designed for post-9/11 veterans living with Post-Traumatic Stress (PTS) and/or Traumatic Brain Injury (TBI). Eligibility requires 

## Summary

This notebook integrated the multi-label classifier (notebook 04) into the RAG pipeline (notebook 03) with three retrieval strategies:

| Strategy | How it works | Pros | Cons |
|----------|-------------|------|------|
| **Basic** | Semantic search only | Simple, fast | May retrieve type-irrelevant results |
| **Filtered** | Semantic search + metadata type filter | Strict type matching | May exclude relevant results if filter is too narrow |
| **Hybrid** | Retrieve 20, re-rank with type-match scoring | Best of both worlds; always returns N results | Slightly more compute; depends on classifier quality |

### What the classifier adds
- **Better retrieval** — services matching detected types get ranked higher
- **Better generation** — LLM knows what types of services to prioritize
- **Graceful degradation** — if classifier detects nothing, falls back to pure semantic search

### Performance notes
- Classifier inference: ~10-20ms (DistilBERT is small)
- ChromaDB query (20 vs 5 results): negligible difference
- Re-ranking 20 candidates: sub-millisecond
- LLM generation dominates total latency regardless of retrieval strategy

### Possible future enhancements
- Tune semantic/type weight ratio with a labeled evaluation set
- Add geographic filtering using `area_served` metadata
- Use classifier confidence to dynamically adjust type weight
- Build a Streamlit/Gradio interface around `ask_case_manager()`